# 1. 环境配置

## 1.1 python 环境准备

In [ ]:
! pip install gradio==6.2.0 openai==2.11.0 dashscope==1.25.4 langchain-classic==1.0.0 langchain==1.2.0 langchain-community==0.4.1 langchain-openai==1.1.6 beautifulsoup4==4.14.3

## 1.2 大模型密钥准备

请根据第一章内容获取相关平台的 API KEY，如若未在系统变量中填入，请将 API_KEY 信息写入以下代码（若已设置请忽略）：

In [ ]:
import os

# os.environ["OPENAI_API_KEY"] = "sk-xxxxxxxx"
# os.environ["DASHSCOPE_API_KEY"] = "sk-yyyyyyyy"

# 2. 文档切分

## 2.1 简介

文档切分是指将一个长文本的 Document 拆成若干个更小的段落（Chunks），每个段落大小适合被大模型理解和向量化处理。

之所以要切分是因为：
- 大模型通常有 Token 长度限制（如 2048 / 4096 tokens），原始文档太长无法直接处理，必须拆分。
- 长文本容易造成语义漂移，而小段文本可以更聚焦地表达一个意思，有利于后续检索和匹配。

即便现在有些技术声称可以处理无限长度的内容，但从经济角度来看，更长的上下文意味着更高的计算成本和费用，因此并不划算。

## 2.2 切分方法

切分的方法有很多种，这里我们主要介绍两种：
- CharacterTextSplitter（基于字数进行划分）
- RecursiveCharacterTextSplitter（基于符号划分）

### 2.2.1 CharacterTextSplitter

比如这里有一段文档内容我们可以让该切分器进行切分：

In [1]:
from langchain_text_splitters import CharacterTextSplitter
some_text = """When writing documents, writers will use document structure to group content. This can convey to the reader, which idea's are related. For example, closely related ideas are in sentances. Similar ideas are in paragraphs. Paragraphs form a document. \n\n  Paragraphs are often delimited with a carriage return or two carriage returns. Carriage returns are the "backslash n" you see embedded in this string. Sentences have a period at the end, but also, have a space. and words are separated by space."""

此时可以设置一个字符切分器，然后每 450 个字符进行一次切分：

In [3]:
c_splitter = CharacterTextSplitter(
    chunk_size=450,
    chunk_overlap=0,
    separator = ' '
)
print(c_splitter.split_text(some_text))

['When writing documents, writers will use document structure to group content. This can convey to the reader, which idea\'s are related. For example, closely related ideas are in sentances. Similar ideas are in paragraphs. Paragraphs form a document. \n\n Paragraphs are often delimited with a carriage return or two carriage returns. Carriage returns are the "backslash n" you see embedded in this string. Sentences have a period at the end, but also,', 'have a space. and words are separated by space.']


### 2.2.2 RecursiveCharacterTextSplitter

除了 CharacterTextSplitter 以外，其实LangChain 里还有很多其他的切分方式，包括下面要介绍的 RecursiveCharacterTextSplitter。

RecursiveCharacterTextSplitter 会更加细致地分割文档，因为它不仅考虑分割后的文本长度，还会兼顾重叠字符。默认情况下， 其使用  ["\n\n", "\n", " ", ""]  四种特殊符号作为分割文本的标记，并按照优先级顺序进行分割：首先尝试在双换行符（ \n\n ）处分割，然后是单换行符（ \n ），接着是空格（ ``），最后在无法找到合适分割点时强制进行分割。

因此，虽然使用  RecursiveCharacterTextSplitter  分割后的文本长度可能与设定的  chunk_size  不完全一致，但它会更倾向于按照句子或段落的形式来分割文本，从而保持更好的可读性和语义连贯性。

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
some_text = """When writing documents, writers will use document structure to group content. This can convey to the reader, which idea's are related. For example, closely related ideas are in sentances. Similar ideas are in paragraphs. Paragraphs form a document. \n\n  Paragraphs are often delimited with a carriage return or two carriage returns. Carriage returns are the "backslash n" you see embedded in this string. Sentences have a period at the end, but also, have a space. and words are separated by space."""

In [5]:
r_splitter = RecursiveCharacterTextSplitter(
    chunk_size=450,
    chunk_overlap=0, 
    separators=["\n\n", "\n", " ", ""]
)
print(r_splitter.split_text(some_text))

["When writing documents, writers will use document structure to group content. This can convey to the reader, which idea's are related. For example, closely related ideas are in sentances. Similar ideas are in paragraphs. Paragraphs form a document.", 'Paragraphs are often delimited with a carriage return or two carriage returns. Carriage returns are the "backslash n" you see embedded in this string. Sentences have a period at the end, but also, have a space. and words are separated by space.']


我们可以尝试真实的对前面网页里的内容进行切分：

In [7]:
from langchain_community.document_loaders import WebBaseLoader
loader = WebBaseLoader("https://zh.d2l.ai/chapter_introduction/index.html")
docs = loader.load()

from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
 chunk_size = 1500,
 chunk_overlap = 150)
splits = text_splitter.split_documents(docs)
print(len(splits))

27


由此我们可以看出，对于自然语言处理的任务来说，一般情况下为了获取每个句子中更多的语义信息，我们通常会使用 RecursiveCharacterTextSplitter 的方式进行文本的分割。

当然文本的分割其实还有很多能够调整的地方，比如说我们可以不仅仅用默认的 separators=["\n\n", "\n", " ", ""]  ，我们可以加入一些别的分割符号，比如 separators=["\n\n", "\n", "(?<=\. )", " ", ""]  ，并且根据文本的不同来进行调整。

除了以上文本处理的方法以外 LangChain 的官方文档中还介绍了关于如何处理结构化文本（Markdown、HTML、JSON、代码...）的切分器。

## 2.3 结构化文本切分器

### 2.3.1 Markdown 文档切分

由于 Markdown 本身就已经是“语义结构化文档”，里内部天然有：
- `# 章节`
- `## 小节`
- `### 子节`

这些比“字符长度”更能代表语义边界。所以不按结构切，相当于浪费信息。
Markdown 切分的核心思想是：“先按“标题结构”分组，再在组内按长度细切。”
比如现在我有一段 Markdown 文本，其内部有最多三级标题：

In [8]:
markdown_document = "# Foo\n\n ## Bar\n\nHi this is Jim\n\nHi this is Joe\n\n ### Boo \n\n Hi this is Lance \n\n ## Baz\n\n Hi this is Molly"

此时假如要对该文本进行切分，我们需要先设置三级标题，并且给出对应的名称：

In [9]:
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]

然后就可以载入 MarkdownHeaderTextSplitter （需要安装依赖 langchain-text-splitters），并将该段文本切分：

In [10]:
from langchain_text_splitters import MarkdownHeaderTextSplitter
markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
md_header_splits = markdown_splitter.split_text(markdown_document)
print(md_header_splits)

[Document(metadata={'Header 1': 'Foo', 'Header 2': 'Bar'}, page_content='Hi this is Jim  \nHi this is Joe'), Document(metadata={'Header 1': 'Foo', 'Header 2': 'Bar', 'Header 3': 'Boo'}, page_content='Hi this is Lance'), Document(metadata={'Header 1': 'Foo', 'Header 2': 'Baz'}, page_content='Hi this is Molly')]


除此之外，在 MarkdownHeaderTextSplitter 中还可以传入其他参数：
- strip_headers（是否保留标题文本到 page_content 中，默认 True 不保留）
- return_each_line（是否“拆到行级”，设置为 True 的话会将第一条 Document 中的 'Hi this is Jim \nHi this is Joe' 进一步拆分为两条 Document）

假如希望对 page_content 里的内容进行更进一步的划分，也可以在使用完 Mardown 拆分器后再使用 RecursiveCharacterTextSplitter 进行切分。

In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=20, chunk_overlap=5)
splits = text_splitter.split_documents(md_header_splits)
print(splits)  

[Document(metadata={'Header 1': 'Foo', 'Header 2': 'Bar'}, page_content='Hi this is Jim'), Document(metadata={'Header 1': 'Foo', 'Header 2': 'Bar'}, page_content='Hi this is Joe'), Document(metadata={'Header 1': 'Foo', 'Header 2': 'Bar', 'Header 3': 'Boo'}, page_content='Hi this is Lance'), Document(metadata={'Header 1': 'Foo', 'Header 2': 'Baz'}, page_content='Hi this is Molly')]


### 2.3.2 HTML 文档切分

对于 HTML 里的文档切分其实也是类似的，但是不一样的是 HTML 不只是“文本”，而是“带语义结构的文档树（DOM）”。 HTML 里有：

- 标题层级（h1 ~ h6）
- 段落（p）
- 列表（ul / ol）
- 表格（table）
- 媒体（img / video / iframe）
- 代码块（pre / code）

这也是为什么假如我们只按字符切，语义会被直接破坏。所以 HTML 切分的核心目标是在“可检索长度”和“语义完整性”之间取得平衡。

在 LangChain 中提供了三种不同的切分方式：
- HTMLHeaderTextSplitter：按 <h1~hN> 标题层级切分 HTML，保留章节结构，适合结构清晰的文档。
- HTMLSectionSplitter：按页面中的“区块结构”切分 HTML，适合结构不规范、来源于网页或爬虫的内容。
- HTMLSemanticPreservingSplitter：在切分时优先保护表格、列表、媒体等语义整体，避免重要结构被拆碎，适合高质量 RAG 场景。

对于 RAG 应用而言，优先选择 HTMLSemanticPreservingSplitter 方式进行切分，因为相比于长度，该方法更看重语义完整性。具体使用方式详见 LangChain 文档。

### 2.3.3 JSON 文档切分
对于 JSON 文档而言，LangChain 中提供了 RecursiveJsonSplitter 方法进行切割，其按 JSON 结构递归切分大型 JSON 对象，尽量保持嵌套结构完整，并控制每一块的字符大小。

它的切分逻辑和“按段落切文本”完全不同，是结构优先，而不是字符串优先：
- 深度优先遍历 JSON
- 优先保持一个对象（dict）不被拆散
- 不会随意切字符串
- 不默认拆 list（因为 list 通常是语义整体）

其最常有三种用法：
- splitter.split_json(json_data)：得到「结构化 JSON 块」

    `[{"openapi": "...", "info": {...}},{"paths": {...}}]`

- splitter.create_documents(texts=[json_data])：将内容切分为 LangChain Document（RAG 场景）

    `page_content='{"openapi": "3.1.0", "info": {"title": "LangSmith", "version": "0.1.0"}, "servers": [{"url": "https://api.smith.langchain.com", "description": "LangSmith API endpoint."}]}'`

- splitter.split_text(json_data)：直接得到 JSON 字符串列表。

更详细的调用指南请查阅 LangChain 文档。

### 2.3.4 代码文档切分
对于代码类文件，LangChain 中也有相关的切分工具，其标不是“按长度切”，而是尽量按“代码结构边界”切，避免把一个函数、类或语句块拆碎。

本质上所有这些代码切分都是基于 RecursiveCharacterTextSplitter ，只是预先帮你准备好了不同语言的「分隔符优先级表」。

这里支持的语言包括 python, cpp, go, java, rust, html, latex, markdown, c 等，这些切分的规则可以通过 Language 模块进行获取：

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, Language

RecursiveCharacterTextSplitter.get_separators_for_language(Language.PYTHON)

['\nclass ', '\ndef ', '\n\tdef ', '\n\n', '\n', ' ', '']

在实际切分时，可以设置对应的语言即可（更多其他语言切分示例请查阅文档内容）：

In [13]:
from langchain_text_splitters import (RecursiveCharacterTextSplitter, Language)

PYTHON_CODE = """
def hello_world():
    print("Hello, World!")

# Call the function
hello_world()
"""

python_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON, chunk_size=50, chunk_overlap=0)
python_docs = python_splitter.create_documents([PYTHON_CODE])
print(python_docs)

[Document(metadata={}, page_content='def hello_world():\n    print("Hello, World!")'), Document(metadata={}, page_content='# Call the function\nhello_world()')]
